# Практика 20 · Vision Transformer

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> ⏱ **Цей зошит навчає 31 маленьку мережу.** Заміряно: **близько девʼяти хвилин**
> (524 і 583 секунди у двох прогонах) на чотирьох ядрах без відеокарти, в один
> потік, на завантаженій машині. Це нормально: предмет теми — саме те, скільки
> трансформер коштує порівняно зі згорткою. Найдовша клітинка — замір 7,
> вісімнадцять прогонів.

Ми зберемо Vision Transformer із нуля й перевіримо кожне твердження лекції числом.
Порядок такий: спершу все, що доводиться **без навчання** (це швидко й цікаво),
потім заміри, які вимагають прогонів.

**Без навчання:**

1. Власна нарізка на патчі проти `unfold` і проти `Conv2d` — доказ, що проєкція
   патча це згортка.
2. Ціна розміру патча: токени, пари уваги, параметри, роздільність.
3. Розклад справжніх ViT-B/16 і ViT-B/32 по частинах.
4. Наш блок енкодера вручну проти `nn.TransformerEncoderLayer`.
5. Норма спереду проти норми ззаду: що лишається від входу після 12 блоків.
6. Головний доказ теми: ViT без позиційного кодування **не бачить** розташування.

**З навчанням:**

7. ViT проти CNN на трьох обсягах даних.
8. CLS-токен проти усереднення по токенах.
9. Задача, де клас визначає положення, — з кодуванням і без.
10. AdamW проти SGD.

In [ ]:
import time
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

# Один потік: на дрібних тензорах він і швидший за чотири, і — головне —
# детермінований. Під кількома потоками додавання float іде в іншому порядку,
# і числа пливуть від прогону до прогону.
torch.set_num_threads(1)

notebook_started = time.perf_counter()

print("torch      :", torch.__version__)
print("numpy      :", np.__version__)
print("потоків CPU:", torch.get_num_threads())

## Дані: ті самі шість фігур

Датасет той самий, що в блоках 2 і 3: шість класів фігур 28×28, згенерованих
формулами. Нічого не завантажується. Шум 0.45, центр гуляє на ±5 пікселів —
робочий варіант складності для цього курсу.

In [ ]:
SHAPE_NAMES = ["коло", "квадрат", "ромб", "кільце", "хрест", "трикутник"]


def draw_shape(kind, rng, size=28, jitter=5, noise=0.45, center=None, radius=None):
    '''Малює одну фігуру заданого класу як масив 28×28 зі значеннями 0..1.'''
    image = np.zeros((size, size), dtype=np.float32)
    if center is None:
        # центр зсуваємо, щоб мережа не завчила одне-єдине положення предмета
        center_y = size / 2 + rng.integers(-jitter, jitter + 1)
        center_x = size / 2 + rng.integers(-jitter, jitter + 1)
    else:
        center_y, center_x = center
    if radius is None:
        radius = rng.integers(5, 9)

    # відстані кожного пікселя від центра — з них складаються всі шість фігур
    yy, xx = np.mgrid[0:size, 0:size]
    dy, dx = yy - center_y, xx - center_x

    if kind == 0:                                    # коло
        image[dy * dy + dx * dx <= radius * radius] = 1.0
    elif kind == 1:                                  # квадрат
        image[(np.abs(dy) <= radius * 0.85) & (np.abs(dx) <= radius * 0.85)] = 1.0
    elif kind == 2:                                  # ромб
        image[np.abs(dy) + np.abs(dx) <= radius] = 1.0
    elif kind == 3:                                  # кільце
        distance = dy * dy + dx * dx
        image[(distance <= radius * radius) & (distance >= (radius - 3) ** 2)] = 1.0
    elif kind == 4:                                  # хрест
        image[(np.abs(dy) <= 2) & (np.abs(dx) <= radius)] = 1.0
        image[(np.abs(dx) <= 2) & (np.abs(dy) <= radius)] = 1.0
    else:                                            # трикутник
        image[(dy >= -radius * 0.8) & (dy <= radius * 0.8)
              & (np.abs(dx) <= (dy + radius * 0.8) * 0.6)] = 1.0

    image += rng.normal(0, noise, image.shape).astype(np.float32)
    return np.clip(image, 0, 1)


def make_dataset(count, rng):
    '''Повертає (count, 1, 28, 28) і (count,). Класи чергуються, тож вони збалансовані.'''
    images = np.zeros((count, 1, 28, 28), dtype=np.float32)
    labels = np.zeros(count, dtype=np.int64)
    for i in range(count):
        kind = i % len(SHAPE_NAMES)
        images[i, 0] = draw_shape(kind, rng)
        labels[i] = kind
    return torch.from_numpy(images), torch.from_numpy(labels)


data_rng = np.random.default_rng(42)
train_x, train_y = make_dataset(2400, data_rng)
test_x, test_y = make_dataset(600, data_rng)

print("навчальних :", tuple(train_x.shape))
print("перевірних :", tuple(test_x.shape))
print("класів     :", len(SHAPE_NAMES), "→ вгадування навмання", round(1 / len(SHAPE_NAMES), 3))
print("яскравість : від", round(float(train_x.min()), 2), "до", round(float(train_x.max()), 2))

## Замір 1 · Ріжемо зображення на патчі власноруч

Найперше, що робить ViT, — перетворює картинку на послідовність векторів.
Напишемо це найпростішим можливим циклом: беремо квадратне вікно P×P, розгортаємо
його в рядок, складаємо рядки в матрицю.

Для 28×28 і патча 4×4 має вийти матриця 49 × 16: сорок девʼять патчів,
у кожному шістнадцять чисел.

In [ ]:
def cut_into_patches(image, patch_size):
    '''Ріже одноканальне зображення на патчі й розгортає кожен у рядок.

    Повертає матрицю (кількість патчів, patch_size * patch_size).
    Порядок обходу — рядками зліва направо, як читають текст.
    '''
    height, width = image.shape
    patches = []
    for row in range(0, height, patch_size):
        for col in range(0, width, patch_size):
            window = image[row:row + patch_size, col:col + patch_size]
            patches.append(window.reshape(-1))     # розгортаємо квадрат у рядок
    return np.stack(patches)


one_image = train_x[0, 0].numpy()
our_patches = cut_into_patches(one_image, 4)

print("зображення  :", one_image.shape)
print("наші патчі  :", our_patches.shape, "— 49 патчів по 16 чисел")
print("перший патч :", np.round(our_patches[0], 2))

## Замір 2 · Наша нарізка = `unfold` = `Conv2d`

Тепер головна чесна деталь теми. Стверджується, що «нарізати + розгорнути +
помножити на спільну матрицю» — це дослівно згортка з ядром P і кроком P.
Перевіримо обидві половини твердження.

Спершу порівняємо саму нарізку з `torch.nn.functional.unfold` — бібліотечною
функцією, яка робить те саме. Потім побудуємо `Conv2d(1, 48, kernel_size=4,
stride=4)`, візьмемо **його ваги**, помножимо на них наші патчі вручну — і
звіримо з тим, що видає сам шар.

In [ ]:
# 1) наша нарізка проти unfold
image_batch = train_x[0:1]                       # (1, 1, 28, 28)
library_patches = F.unfold(image_batch, kernel_size=4, stride=4)[0].T.numpy()

assert np.allclose(our_patches, library_patches), "нарізка розійшлася з unfold!"
print("наша нарізка == unfold  :", np.array_equal(our_patches, library_patches),
      " макс |різниця| =", float(np.abs(our_patches - library_patches).max()))

# 2) наше множення проти Conv2d
torch.manual_seed(0)
patch_projection = nn.Conv2d(1, 48, kernel_size=4, stride=4)

# ваги згортки — це і є матриця проєкції, лише складена як (48, 1, 4, 4)
weight_matrix = patch_projection.weight.detach().numpy().reshape(48, 16)
bias_vector = patch_projection.bias.detach().numpy()

by_hand = our_patches @ weight_matrix.T + bias_vector          # (49, 48)
by_conv = patch_projection(image_batch).flatten(2).transpose(1, 2).detach().numpy()[0]

assert np.allclose(by_hand, by_conv, atol=1e-5), "проєкція розійшлася зі згорткою!"
print("наше множення == Conv2d :", np.allclose(by_hand, by_conv, atol=1e-5),
      " макс |різниця| =", float(np.abs(by_hand - by_conv).max()))
print()
print("✅ проєкція патча — це згортка з ядром 4 і кроком 4, а не «щось схоже»")
print("   ваг у проєкції: 4 × 4 × 1 × 48 + 48 =",
      4 * 4 * 1 * 48 + 48,
      "· у шарі:", sum(p.numel() for p in patch_projection.parameters()))

## Замір 3 · Скільки коштує розмір патча

Розмір патча — головна ручка ціни. Порахуємо для нашого зображення 28×28 чотири
величини одразу: скільки виходить токенів, скільки пар уваги рахує один шар
(це квадрат кількості токенів), скільки ваг у матриці проєкції і яка лишається
роздільність.

In [ ]:
TOKEN_DIM = 48

print("патч | токенів | пар уваги | ваг у проєкції | роздільність")
print("-" * 62)
for patch_size in (2, 4, 7, 14):
    grid = 28 // patch_size
    tokens = grid * grid
    attention_pairs = tokens * tokens
    projection_weights = patch_size * patch_size * 1 * TOKEN_DIM + TOKEN_DIM
    print(f"{patch_size:>7} | {tokens:>7} | {attention_pairs:>9} | "
          f"{projection_weights:>14} | {grid}×{grid}")

print()
print("патч 2 проти патча 4: пар уваги більше в",
      round((14 * 14) ** 2 / (7 * 7) ** 2), "разів,",
      "а ваг у проєкції менше в", round((4 * 4 * 48 + 48) / (2 * 2 * 48 + 48), 1), "раза")

## Замір 4 · Розклад справжнього ViT-B по частинах

Тепер доросла модель. `torchvision` збирає ViT-B двох різновидів — з патчем 16
і з патчем 32. Обидва працюють із зображенням 224×224, обидва мають розмірність
токена 768 і дванадцять блоків. Різниця рівно одна: розмір патча.

`weights=None` означає, що нічого не завантажується з мережі — це чиста
арифметика будови.

In [ ]:
def vit_breakdown(model):
    '''Розкладає параметри ViT з torchvision на змістовні частини.'''
    encoder_layers = sum(p.numel() for p in model.encoder.layers.parameters())
    return {
        "усього":              sum(p.numel() for p in model.parameters()),
        "токенів (із CLS)":    model.encoder.pos_embedding.shape[1],
        "проєкція патча":     sum(p.numel() for p in model.conv_proj.parameters()),
        "позиційне кодування": model.encoder.pos_embedding.numel(),
        "CLS-токен":           model.class_token.numel(),
        "шари енкодера":       encoder_layers,
        "фінальний LayerNorm": sum(p.numel() for p in model.encoder.ln.parameters()),
        "класифікатор":        sum(p.numel() for p in model.heads.parameters()),
    }


b16 = vit_breakdown(models.vit_b_16(weights=None))
b32 = vit_breakdown(models.vit_b_32(weights=None))

print(f"{'частина':<22}{'ViT-B/16':>14}{'ViT-B/32':>14}")
print("-" * 50)
for key in b16:
    print(f"{key:<22}{b16[key]:>14,}{b32[key]:>14,}".replace(",", " "))

print()
assert b16["шари енкодера"] == b32["шари енкодера"]
print("✅ шари енкодера збігаються ДО БІТА:", f"{b16['шари енкодера']:,}".replace(",", " "))
print("   проєкція /32 перевіряється рахунком: 32 × 32 × 3 × 768 + 768 =",
      32 * 32 * 3 * 768 + 768)
assert b32["проєкція патча"] == 32 * 32 * 3 * 768 + 768
print()
print("ViT-B/32 має БІЛЬШЕ параметрів на",
      f"{b32['усього'] - b16['усього']:,}".replace(",", " "),
      "— і при цьому вчетверо менше токенів, тобто рахується швидше")

# заразом порахуємо дві моделі, про які згадує кінець лекції
print()
print("для порівняння, теж без завантаження ваг:")
print("  swin_t        :", sum(p.numel() for p in models.swin_t(weights=None).parameters()))
print("  convnext_tiny :", sum(p.numel() for p in models.convnext_tiny(weights=None).parameters()))

## Наша крихітна модель

Тепер зберемо ViT для нашої задачі: патч 4×4, 49 токенів, розмірність 48,
два блоки, чотири голови. Прапорець `use_pos` дозволяє вимкнути позиційне
кодування, `pool` — обрати між CLS-токеном і усередненням по токенах.
Це знадобиться для замірів далі.

In [ ]:
class TinyViT(nn.Module):
    '''Vision Transformer для 28×28: проєкція патча → CLS → енкодер → класифікатор.'''

    def __init__(self, patch=4, dim=48, depth=2, heads=4, feedforward=96,
                 n_classes=6, use_pos=True, pool="cls"):
        super().__init__()
        self.patch = patch
        self.pool = pool
        n_tokens = (28 // patch) ** 2

        # згортка з ядром = кроку і є проєкцією патча (замір 2)
        self.to_patch = nn.Conv2d(1, dim, kernel_size=patch, stride=patch)

        # CLS — зайвий токен, який не відповідає жодному патчу
        self.cls = nn.Parameter(torch.zeros(1, 1, dim)) if pool == "cls" else None
        total_tokens = n_tokens + (1 if pool == "cls" else 0)

        # позиційне кодування: по вектору на кожне місце в послідовності
        self.pos = nn.Parameter(torch.zeros(1, total_tokens, dim)) if use_pos else None
        if use_pos:
            nn.init.trunc_normal_(self.pos, std=0.02)

        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=feedforward,
            batch_first=True, norm_first=True, dropout=0.0)
        self.encoder = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, n_classes)

    def forward(self, x):
        tokens = self.to_patch(x).flatten(2).transpose(1, 2)     # (B, 49, dim)
        if self.cls is not None:
            tokens = torch.cat([self.cls.expand(tokens.shape[0], -1, -1), tokens], dim=1)
        if self.pos is not None:
            tokens = tokens + self.pos
        tokens = self.norm(self.encoder(tokens))
        summary = tokens[:, 0] if self.pool == "cls" else tokens.mean(1)
        return self.head(summary)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


tiny = TinyViT()
parts = {
    "проєкція патча":     sum(p.numel() for p in tiny.to_patch.parameters()),
    "CLS-токен":           tiny.cls.numel(),
    "позиційне кодування": tiny.pos.numel(),
    "два блоки енкодера":  sum(p.numel() for p in tiny.encoder.parameters()),
    "фінальний LayerNorm": sum(p.numel() for p in tiny.norm.parameters()),
    "класифікатор":        sum(p.numel() for p in tiny.head.parameters()),
}
for key, value in parts.items():
    print(f"{key:<22}{value:>8}")
print("-" * 30)
print(f"{'усього':<22}{count_params(tiny):>8}")
assert sum(parts.values()) == count_params(tiny)
print()
print("на енкодер припадає", round(100 * parts["два блоки енкодера"] / count_params(tiny), 1), "% ваг")

## Замір 5 · Блок енкодера, написаний вручну

У блоці немає магії. При нормі спереду він робить рівно чотири речі:

```
x = x + Увага(LayerNorm(x))
x = x + MLP(LayerNorm(x))
```

Напишемо це чотирма рядками, візьмемо ваги бібліотечного шару й перевіримо, що
результат збігається. Якщо збігається — значить, ми справді знаємо, що всередині.

In [ ]:
def encoder_block_by_hand(layer, x):
    '''Ручний прохід блоку з нормою спереду (norm_first=True).'''
    normed = layer.norm1(x)
    attended, _ = layer.self_attn(normed, normed, normed, need_weights=False)
    x = x + attended                                  # залишковий звʼязок, тема 14

    normed = layer.norm2(x)
    mlp_out = layer.linear2(layer.activation(layer.linear1(normed)))
    return x + mlp_out


torch.manual_seed(0)
probe_input = torch.randn(2, 50, 48)
one_layer = tiny.encoder.layers[0]
one_layer.eval()

with torch.no_grad():
    ours = encoder_block_by_hand(one_layer, probe_input)
    library = one_layer(probe_input)

difference = float((ours - library).abs().max())
assert np.allclose(ours.numpy(), library.numpy(), atol=1e-6), "блок розійшовся!"
print("наш блок == nn.TransformerEncoderLayer :", torch.equal(ours, library))
print("макс |різниця|                         :", difference)
print()
print("ваги одного блоку при розмірності 48:")
print("  увага (запити, ключі, значення, вихід):",
      sum(p.numel() for p in one_layer.self_attn.parameters()))
print("  MLP 48 → 96 → 48                      :",
      sum(p.numel() for p in one_layer.linear1.parameters())
      + sum(p.numel() for p in one_layer.linear2.parameters()))
print("  два LayerNorm                         :",
      sum(p.numel() for p in one_layer.norm1.parameters())
      + sum(p.numel() for p in one_layer.norm2.parameters()))
print("  разом                                 :",
      sum(p.numel() for p in one_layer.parameters()))

## Замір 6 · Норма спереду проти норми ззаду

Два варіанти розміщення LayerNorm відрізняються тим, чи стоїть нормалізація
**на магістралі** залишкового потоку, чи **на гілці**. Подивимось, що з цього
виходить: пропустимо той самий випадковий вхід крізь стос із дванадцяти
ненавчених блоків і після кожного поміряємо косинус між тим, що тече зараз,
і початковим входом.

Косинус близький до 1 означає «початковий сигнал іще на місці», близький до 0 —
«від нього нічого не лишилось».

In [ ]:
def residual_trace(norm_first, depth=12, dim=48, seed=0):
    '''Косинус між входом стосу й потоком після кожного блоку.'''
    torch.manual_seed(seed)
    layer = nn.TransformerEncoderLayer(dim, 4, 96, batch_first=True,
                                       norm_first=norm_first, dropout=0.0)
    stack = nn.TransformerEncoder(layer, num_layers=depth)
    stack.eval()

    torch.manual_seed(123)                      # той самий вхід для обох варіантів
    start = torch.randn(8, 50, dim)
    flowing = start
    similarity = []
    with torch.no_grad():
        for block in stack.layers:
            flowing = block(flowing)
            similarity.append(float(
                F.cosine_similarity(start.flatten(1), flowing.flatten(1)).mean()))
    return similarity


pre_norm = residual_trace(True)
post_norm = residual_trace(False)

print("після блоку | норма спереду | норма ззаду")
print("-" * 44)
for i in (0, 3, 7, 11):
    print(f"{i + 1:>11} | {pre_norm[i]:>13.3f} | {post_norm[i]:>11.3f}")
print()
print("після дванадцяти блоків від входу лишилось:",
      round(pre_norm[-1], 3), "спереду проти", round(post_norm[-1], 3), "ззаду —",
      "у", round(pre_norm[-1] / post_norm[-1], 1), "раза більше")
print("Саме тому сучасні реалізації беруть norm_first=True.")

## Згорткова мережа для порівняння

Щоб порівняння було чесним, потрібна проста згорткова мережа на тих самих даних.
Три згортки 3×3 із батчнормом, два пулінги, глобальне усереднення, лінійний шар.
Ваг у ній буде у вісім разів менше, ніж у нашого ViT.

In [ ]:
class TinyCNN(nn.Module):
    '''Три згортки 3×3 → глобальне усереднення → лінійний шар.'''

    def __init__(self, n_classes=6, c1=8, c2=16, c3=24):
        super().__init__()
        self.body = nn.Sequential(
            # bias=False, бо батчнорм одразу за згорткою однаково відніме будь-який зсув
            nn.Conv2d(1, c1, 3, padding=1, bias=False), nn.BatchNorm2d(c1), nn.ReLU(),
            nn.MaxPool2d(2),                                        # 28×28 → 14×14
            nn.Conv2d(c1, c2, 3, padding=1, bias=False), nn.BatchNorm2d(c2), nn.ReLU(),
            nn.MaxPool2d(2),                                        # 14×14 → 7×7
            nn.Conv2d(c2, c3, 3, padding=1, bias=False), nn.BatchNorm2d(c3), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),                                # глобальне усереднення
        )
        self.head = nn.Linear(c3, n_classes)

    def forward(self, x):
        return self.head(self.body(x).flatten(1))


print("ваг у ViT :", count_params(TinyViT()))
print("ваг у CNN :", count_params(TinyCNN()))
print("ViT важчий у", round(count_params(TinyViT()) / count_params(TinyCNN()), 1), "раза")

## Найважливіший доказ теми — і він не потребує навчання

Перш ніж щось навчати, доведемо головне твердження розділу 9 лекції алгеброю.

Побудуємо задачу «де фігура»: чотири класи — чотири чверті кадру. Форма фігури
випадкова, тож про клас вона не говорить нічого; клас визначає **лише**
розташування. Центри стоять у точках (8, 8), (8, 20), (20, 8) і (20, 20) —
зсув між чвертями дорівнює 12 пікселям, тобто **рівно трьом патчам по чотири**.

Це зроблено навмисне. Якщо зсув кратний патчу, то той самий предмет у різних
чвертях дає **той самий набір патчів**, просто розкладений по інших місцях
сітки. А модель без позиційного кодування бачить не сітку, а невпорядкований
мішок токенів.

In [ ]:
QUADRANT_CENTERS = [(8, 8), (8, 20), (20, 8), (20, 20)]
QUADRANT_NAMES = ["ліва верхня", "права верхня", "ліва нижня", "права нижня"]


def make_where_dataset(count, rng):
    '''Клас = чверть, у якій стоїть фігура. Форма фігури про клас не говорить нічого.'''
    images = np.zeros((count, 1, 28, 28), dtype=np.float32)
    labels = np.zeros(count, dtype=np.int64)
    for i in range(count):
        quadrant = i % 4
        shape_kind = int(rng.integers(0, len(SHAPE_NAMES)))   # форма випадкова
        radius = int(rng.integers(4, 7))
        images[i, 0] = draw_shape(shape_kind, rng,
                                  center=QUADRANT_CENTERS[quadrant], radius=radius)
        labels[i] = quadrant
    return torch.from_numpy(images), torch.from_numpy(labels)


check_rng = np.random.default_rng(0)
# одна й та сама фігура в чотирьох чвертях, шуму немає — щоб зсув був точний
same_shape = np.stack([draw_shape(0, check_rng, center=c, radius=5, noise=0.0)
                       for c in QUADRANT_CENTERS])[:, None]
same_shape = torch.from_numpy(same_shape)

print("чотири картинки, однакові з точністю до зсуву на 12 пікселів = 3 патчі")
print("яскравостей у кожній однаково:",
      [int(round(float(same_shape[i].sum()))) for i in range(4)])

Тепер подамо ці чотири картинки **ненавченим** моделям і подивимось, наскільки
відрізняються їхні виходи. Якщо модель фізично не бачить розташування, різниця
буде на рівні похибки float32 — тобто нуль.

Заразом перевіримо згорткову мережу, і не одну: звичайну (доповнення країв
нулями) і таку саму з циклічним доповненням. Це відповість на питання, звідки
CNN із глобальним усередненням узагалі бере інформацію про положення.

In [ ]:
class CircularCNN(TinyCNN):
    '''Та сама CNN, але краї доповнюються не нулями, а протилежним краєм.'''

    def __init__(self, n_classes=4):
        super().__init__(n_classes=n_classes)
        for module in self.body:
            if isinstance(module, nn.Conv2d):
                module.padding_mode = "circular"


def spread_over_quadrants(model, batch):
    '''Наскільки виходи моделі відрізняються між чотирма чвертями.'''
    model.eval()
    with torch.no_grad():
        out = model(batch)
    return float((out - out[0]).abs().max())


torch.manual_seed(0)
checks = [
    ("ViT без позиційного",     TinyViT(n_classes=4, use_pos=False)),
    ("ViT із позиційним",       TinyViT(n_classes=4, use_pos=True)),
    ("CNN, доповнення нулями",  TinyCNN(n_classes=4)),
    ("CNN, доповнення циклічне", CircularCNN(n_classes=4)),
]

print("модель (ненавчена)          | макс різниця виходів між чвертями")
print("-" * 64)
for name, model in checks:
    print(f"{name:<27} | {spread_over_quadrants(model, same_shape):.10f}")

print()
print("✅ ViT без позиційного кодування видає на чотирьох різних картинках")
print("   ОДНАКОВИЙ вихід. Ніяке навчання цього не виправить.")
print("✅ CNN розрізняє чверті — але тільки поки краї доповнюються нулями.")
print("   З циклічним доповненням різниця теж зникає: джерело позиції — саме краї.")

## Заміри з навчанням

Далі йдуть прогони. Спершу спільна функція навчання: **AdamW**, тридцять епох,
батч 64, швидкість навчання 0.003. Один рецепт на всі моделі — інакше ми
порівнювали б рецепти, а не архітектури (урок теми 18).

In [ ]:
def train_and_test(model, x_train, y_train, x_test, y_test,
                   epochs=30, lr=3e-3, batch=64, seed=0, optimizer="adamw"):
    '''Навчає модель і повертає (точність на перевірці, секунди).'''
    torch.manual_seed(seed)
    if optimizer == "adamw":
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    else:
        opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    loss_function = nn.CrossEntropyLoss()
    shuffler = torch.Generator().manual_seed(seed)

    started = time.perf_counter()
    for _ in range(epochs):
        model.train()
        order = torch.randperm(len(x_train), generator=shuffler)
        for start in range(0, len(x_train), batch):
            batch_idx = order[start:start + batch]
            opt.zero_grad()
            loss_function(model(x_train[batch_idx]), y_train[batch_idx]).backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        accuracy = (model(x_test).argmax(1) == y_test).float().mean().item()
    return accuracy, time.perf_counter() - started


# швидка перевірка, що функція жива: одна епоха на маленькій вибірці
torch.manual_seed(0)
warmup_accuracy, warmup_seconds = train_and_test(
    TinyCNN(), train_x[:150], train_y[:150], test_x, test_y, epochs=1)
print(f"пробний прогін: точність {warmup_accuracy:.3f} за {warmup_seconds:.1f} с")
print("(одна епоха на 150 прикладах — від неї нічого не очікуємо, це перевірка коду)")

## Замір 7 · ViT проти CNN на трьох обсягах даних

Головне порівняння теми. Три моделі — згорткова, ViT і ViT без позиційного
кодування — на 150, 600 і 2400 прикладах, по два зерна на кожну точку.

⚠️ Це найдовша клітинка зошита: вісімнадцять прогонів, близько шести хвилин.

In [ ]:
DATA_SIZES = [150, 600, 2400]
SEEDS = [0, 1]

comparison = {}
kept_vit = {}            # збережемо навчену модель, щоб потім зняти з неї карти уваги

for size in DATA_SIZES:
    for name, build in [("CNN", lambda: TinyCNN()),
                        ("ViT", lambda: TinyViT(use_pos=True)),
                        ("ViT без позиційного", lambda: TinyViT(use_pos=False))]:
        runs = []
        for seed in SEEDS:
            torch.manual_seed(seed)
            model = build()
            accuracy, _ = train_and_test(model, train_x[:size], train_y[:size],
                                         test_x, test_y, seed=seed)
            runs.append(accuracy)
            if size == 600 and name == "ViT" and seed == 0:
                kept_vit["model"] = model
        comparison[(size, name)] = runs

MODEL_NAMES = ("CNN", "ViT", "ViT без позиційного")

print("прикладів | модель               | два прогони     | середнє | розкид")
print("-" * 72)
for size in DATA_SIZES:
    for name in MODEL_NAMES:
        runs = comparison[(size, name)]
        values = " · ".join(f"{v:.3f}" for v in runs)
        print(f"{size:>9} | {name:<20} | {values} | {np.mean(runs):>7.3f} | "
              f"{max(runs) - min(runs):.3f}")

print()
print("прикладів |     CNN |     ViT | ViT без позиційного")
print("-" * 52)
for size in DATA_SIZES:
    row = [np.mean(comparison[(size, n)]) for n in MODEL_NAMES]
    print(f"{size:>9} | {row[0]:>7.3f} | {row[1]:>7.3f} | {row[2]:>19.3f}")

print()
gap_small = np.mean(comparison[(150, "CNN")]) - np.mean(comparison[(150, "ViT")])
gap_large = np.mean(comparison[(2400, "CNN")]) - np.mean(comparison[(2400, "ViT")])
print(f"розрив CNN − ViT: {gap_small:.3f} на 150 прикладах, {gap_large:.3f} на 2400")
print("Згортка виграє на кожному обсязі. За шістнадцятикратного зростання даних")
print(f"розрив скоротився всього на {gap_small - gap_large:.3f} — це не «наздоганяння».")
print()
print("І важливе застереження про останній стовпчик: різниця між ViT із кодуванням")
print("і без нього менша за розкид від зерна скрізь, крім 2400 прикладів:")
for size in DATA_SIZES:
    with_pos = np.mean(comparison[(size, "ViT")])
    without = np.mean(comparison[(size, "ViT без позиційного")])
    spread = max(max(comparison[(size, n)]) - min(comparison[(size, n)])
                 for n in ("ViT", "ViT без позиційного"))
    verdict = "більша за розкид" if abs(without - with_pos) > spread else "тоне в розкиді"
    print(f"  {size:>4}: різниця {without - with_pos:+.3f}, розкид {spread:.3f} — {verdict}")

## Замір 8 · CLS-токен проти усереднення

Навіщо ViT додає пʼятдесятий токен, якщо можна просто усереднити всі 49?
Перевіримо це на 600 прикладах, по три зерна на варіант. Два прогони з CLS у нас
уже є із заміру 7 — беремо їх і доганяємо третім зерном.

In [ ]:
cls_runs = list(comparison[(600, "ViT")])          # зерна 0 і 1 уже пораховані
torch.manual_seed(2)
accuracy, _ = train_and_test(TinyViT(pool="cls"), train_x[:600], train_y[:600],
                             test_x, test_y, seed=2)
cls_runs.append(accuracy)

mean_runs = []
for seed in (0, 1, 2):
    torch.manual_seed(seed)
    accuracy, _ = train_and_test(TinyViT(pool="mean"), train_x[:600], train_y[:600],
                                 test_x, test_y, seed=seed)
    mean_runs.append(accuracy)

print("як зводимо 49 токенів в один | три прогони          | середнє | розкид")
print("-" * 74)
for label, runs in [("CLS-токен", cls_runs), ("середнє по токенах", mean_runs)]:
    values = " · ".join(f"{v:.3f}" for v in runs)
    print(f"{label:<28} | {values} | {np.mean(runs):>7.3f} | {max(runs) - min(runs):.3f}")

print()
print("різниця середніх :", round(abs(np.mean(cls_runs) - np.mean(mean_runs)), 3))
print("більший розкид   :", round(max(max(cls_runs) - min(cls_runs),
                                      max(mean_runs) - min(mean_runs)), 3))
print("зайвих ваг за CLS:", TinyViT(pool="cls").cls.numel())
print()
print("Різниця менша за розкид — отже, на нашій задачі CLS не дає нічого.")
print("Він коштує 48 ваг і не окупає їх (правило з теми 18).")

## Замір 9 · Куди дивиться CLS-токен

Знімемо ваги уваги від CLS до кожного з 49 патчів у моделі, яку ми вже навчили
в замірі 7. Нового навчання тут немає — лише один прохід уперед із
`need_weights=True`.

Рівномірна увага дала б кожному патчу 1/49 = 0.0204. Порівнюємо з цим числом.

In [ ]:
def cls_attention_maps(model, batch):
    '''Ваги уваги від CLS до патчів, по шарах і головах: список (B, голів, 49).'''
    tokens = model.to_patch(batch).flatten(2).transpose(1, 2)
    tokens = torch.cat([model.cls.expand(tokens.shape[0], -1, -1), tokens], dim=1)
    tokens = tokens + model.pos

    maps = []
    for layer in model.encoder.layers:
        normed = layer.norm1(tokens)
        attended, weights = layer.self_attn(normed, normed, normed,
                                            need_weights=True, average_attn_weights=False)
        maps.append(weights[:, :, 0, 1:].detach())     # рядок CLS, без нього самого
        tokens = tokens + attended
        normed = layer.norm2(tokens)
        tokens = tokens + layer.linear2(layer.activation(layer.linear1(normed)))
    return maps


trained_vit = kept_vit["model"]
trained_vit.eval()

# точність саме цієї моделі — щоб було з чим звіряти карти
with torch.no_grad():
    kept_accuracy = (trained_vit(test_x).argmax(1) == test_y).float().mean().item()

uniform = 1 / 49
print("модель: ViT на 600 прикладах, зерно 0, точність", round(kept_accuracy, 3))
print("рівномірна увага дала б кожному патчу:", round(uniform, 4))
print()
print("фігура  | шар | голова | найбільша вага | у скільки разів > рівномірної")
print("-" * 72)
for shape_index in (0, 3, 4):                 # коло, кільце, хрест
    example = int((test_y == shape_index).nonzero()[0].item())
    maps = cls_attention_maps(trained_vit, test_x[example:example + 1])
    for layer_index, layer_maps in enumerate(maps, 1):
        for head in range(layer_maps.shape[1]):
            peak = float(layer_maps[0, head].max())
            print(f"{SHAPE_NAMES[shape_index]:<7} | {layer_index:>3} | {head + 1:>6} | "
                  f"{peak:>14.4f} | {peak / uniform:>29.1f}")
    if shape_index == 0:
        first_layer_peak = float(maps[0][0].max())
        second_layer_peak = float(maps[1][0].max())

print()
print("для кола: макс по першому шару", round(first_layer_peak, 4),
      "· по другому", round(second_layer_peak, 4))
print("У першому шарі окремі голови вибирають майже один патч,")
print("у другому увага розмазується по кількох.")

## Замір 10 · Задача, де положення визначає клас

Тепер навчимо ті самі моделі на задачі «де фігура». Чотири класи, вгадування
навмання дає 0.25. Три зерна на кожен варіант ViT.

Ми вже довели алгеброю, що ViT без позиційного кодування має застрягти рівно на
рівні вгадування. Подивимось, чи так воно буде насправді.

In [ ]:
where_rng = np.random.default_rng(7)
where_train_x, where_train_y = make_where_dataset(600, where_rng)
where_test_x, where_test_y = make_where_dataset(400, where_rng)

where_results = {}
for name, build, seeds in [
        ("ViT із позиційним",  lambda: TinyViT(n_classes=4, use_pos=True),  (0, 1, 2)),
        ("ViT без позиційного", lambda: TinyViT(n_classes=4, use_pos=False), (0, 1, 2)),
        ("CNN",                 lambda: TinyCNN(n_classes=4),                (0,))]:
    runs = []
    for seed in seeds:
        torch.manual_seed(seed)
        accuracy, _ = train_and_test(build(), where_train_x, where_train_y,
                                     where_test_x, where_test_y, seed=seed)
        runs.append(accuracy)
    where_results[name] = runs

print("модель                | «яка фігура» | «де фігура» | прогони на «де»")
print("-" * 74)
what_lookup = {"ViT із позиційним": "ViT",
               "ViT без позиційного": "ViT без позиційного",
               "CNN": "CNN"}
for name, runs in where_results.items():
    what = np.mean(comparison[(600, what_lookup[name])])
    values = " · ".join(f"{v:.3f}" for v in runs)
    print(f"{name:<21} | {what:>12.3f} | {np.mean(runs):>11.3f} | {values}")

print()
print("вгадування навмання: 0.167 на «яка фігура», 0.250 на «де фігура»")
print()
without = np.mean(where_results["ViT без позиційного"])
print(f"ViT без позиційного на задачі про розташування: {without:.3f}")
if abs(without - 0.25) < 0.03:
    print("✅ гіпотеза підтвердилась: це рівно рівень вгадування, моделі немає")
else:
    print("⚠️ гіпотеза не підтвердилась: модель щось таки вивчила, розберись чому")
print()
print("Правило: позиційне кодування допомагає рівно тоді, коли положення")
print("визначає клас. На задачі про вміст воно коштує ваг і не дає нічого.")

## Замір 11 · AdamW проти SGD

Протокол цього блоку — AdamW: вважається, що на SGD трансформери навчаються
погано. Перевіримо це твердження на нашій моделі, замість того щоб переказувати.

In [ ]:
sgd_results = {}
for lr in (3e-3, 3e-2):
    torch.manual_seed(0)
    accuracy, _ = train_and_test(TinyViT(), train_x[:600], train_y[:600],
                                 test_x, test_y, seed=0, lr=lr, optimizer="sgd")
    sgd_results[lr] = accuracy

print("оптимізатор          | швидкість навчання | точність")
print("-" * 52)
print(f"{'AdamW':<20} | {0.003:>18} | {np.mean(cls_runs):>8.3f}  (середнє з трьох зерен)")
for lr, accuracy in sgd_results.items():
    print(f"{'SGD, момент 0.9':<20} | {lr:>18} | {accuracy:>8.3f}")

print()
print("На нашій крихітній моделі SGD не гірший за AdamW.")
print("Правило про AdamW здобуте на великих трансформерах, і два шари з 41 574")
print("вагами — надто мала мережа, щоб на ній це правило перевіряти.")
print("Ми лишаємо AdamW як протокол блоку, але число наводимо чесно.")

## Що ми довели

| Твердження | Чим доведено |
|---|---|
| проєкція патча — це згортка | `np.allclose` з `Conv2d`, різниця на рівні float32 |
| розмір патча міняє лише вхід | шари енкодера ViT-B/16 і B/32 збігаються до біта |
| ViT-B/32 більший, але швидший | більше параметрів, учетверо менше токенів |
| блок енкодера — чотири рядки | ручний прохід збігається з бібліотечним |
| норму беруть спереду | після 12 блоків від входу лишається у 8 разів більше |
| на малих даних згортка виграє | CNN попереду на всіх трьох обсягах |
| CLS не є джерелом сили ViT | різниця з усередненням менша за розкид від зерна |
| без позиційного немає геометрії | однаковий вихід на чотирьох різних картинках |

---

## Завдання

### 🟢 Рівень 1

Поміняй розмір патча нашого ViT із 4 на 7 (сітка стане 4×4, токенів 16) і
навчи модель на 600 прикладах. Порахуй, як змінилися: кількість ваг, час одного
навчання й точність. Поясни словами, чому кількість ваг **зросла**, хоч токенів
стало менше.

### 🟡 Рівень 2

Побудуй третю задачу — таку, де клас залежить і від вмісту, і від положення
одночасно. Наприклад: два класи — «коло вгорі, квадрат унизу» проти «квадрат
угорі, коло внизу». Навчи ViT із позиційним кодуванням і без нього. Що станеться
з варіантом без кодування і чому саме таке число?

### 🔴 Рівень 3

Збери блок енкодера **з нуля**: власна багатоголова увага з матрицями запитів,
ключів і значень, власний MLP, власні LayerNorm. Скопіюй у свій блок ваги
бібліотечного `nn.TransformerEncoderLayer` і доведи `assert np.allclose(...)`,
що на однаковому вході обидва дають однаковий вихід. Підказка: у PyTorch
матриці запитів, ключів і значень зберігаються **однією** склеєною матрицею
`in_proj_weight` розміром (3 × dim, dim).

In [ ]:
print(f"зошит виконано за {time.perf_counter() - notebook_started:.0f} секунд")